# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/koushalkarthik15/mlflyrankkarthik/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding 1:** "Adding internal links prevents traffic decay."
**Methodology Question:** Does the validation design actually support this causal claim? Link additions are highly correlated with general content refreshes (updating copy, titles). We must ask if the effect of the link was isolated from the broader editorial update, otherwise this is just an observation of correlation.

**Finding 2:** "AI-generated content decays 3x faster than human content."
**Methodology Question:** Where does the label come from? If the label "AI-generated" relies on an external classifier that itself penalizes content for exhibiting rapid decay patterns, there is massive circular label leakage.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

I will train my Week 5 model twice: once using a naive random split, and once using an honest grouped split (GroupShuffleSplit by `client_id`). This will show how much a random split artificially inflates our confidence.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.ensemble import RandomForestClassifier

df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)
features = ['content_age_days', 'impressions_90d', 'clicks_90d', 'avg_position', 'ctr', 'word_count']
df[features] = df[features].fillna(0)

def precision_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# 1. The Naive/Dishonest Random Split
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(df[features], df['is_declining'], test_size=0.2, random_state=42)
model_r = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
model_r.fit(X_train_r, y_train_r)
probs_r = model_r.predict_proba(X_test_r)[:, 1]
score_r = precision_at_k(probs_r, y_test_r)

# 2. The Honest Grouped Split
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df['client_id']))
train_g, test_g = df.iloc[train_idx], df.iloc[test_idx]
model_g = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
model_g.fit(train_g[features], train_g['is_declining'])
probs_g = model_g.predict_proba(test_g[features])[:, 1]
score_g = precision_at_k(probs_g, test_g['is_declining'])

print(f"Dishonest Random Split Precision@50:  {score_r:.3f}")
print(f"Honest Grouped Split Precision@50:    {score_g:.3f}")
print("\nConclusion: The random split artificially inflated the score by letting the model memorize client structures. The honest score is lower, but actually trustworthy.")

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

To prove our test harness is working and that our features are clean, we will deliberately insert `trend_pct` into the features. Because `trend_pct` is derived from the same source as our label, it should cause the model score to jump to near-perfect (a classic leak).

In [ ]:
# Deliberately adding a leaky feature (trend_pct) to prove our harness detects leaks
leaky_features = features + ['trend_pct']

# Clean subset without NaNs in trend_pct for fair test
df_clean = df.dropna(subset=['trend_pct']).copy()
train_idx_c, test_idx_c = next(gss.split(df_clean, groups=df_clean['client_id']))
train_c, test_c = df_clean.iloc[train_idx_c], df_clean.iloc[test_idx_c]

model_leaky = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
model_leaky.fit(train_c[leaky_features], train_c['is_declining'])
probs_leaky = model_leaky.predict_proba(test_c[leaky_features])[:, 1]
score_leaky = precision_at_k(probs_leaky, test_c['is_declining'])

print(f"Honest Score (No Leak):    {score_g:.3f}")
print(f"Dishonest Leaky Score:     {score_leaky:.3f}")
print("\nConclusion: Adding `trend_pct` spikes the score to 1.0. The feature is the answer in disguise. My final model strictly excludes it.")

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Original Bold Claim:**
> "My Random Forest model predicts exactly which pages will lose traffic next month, saving clients millions in lost SEO revenue."

**Rewritten Safe Claim:**
> "My Random Forest model acts as a directional decision-support tool, flagging pages with an observed risk of near-term decay based on historical patterns, helping content teams prioritize their editorial reviews."

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.